# Análise da Produção Agrícola no Brasil (2015–2024)

**Projeto G2 — Tema 29** ·
**Disciplina — Linguagens de Programação**
**Aluno: Rafael Rigo de Oliveira**
**Professor: Alexandre Neves Louzada**

---

## 1. Introdução

Este notebook demonstra a análise exploratória da produção agrícola brasileira entre **2015 e 2024**, usando o arquivo `simulacao_producao_agricola_brasil.csv`. Ele é o complemento do dashboard em **Streamlit** (`app.py`)

### Objetivos
- Analisar a **produtividade** agrícola e identificar as **culturas** mais relevantes;
- Comparar **regiões e estados** produtores;
- Investigar a relação entre **clima** (chuva e temperatura) e produtividade;
- Avaliar o **crescimento** do setor e as **exportações** ao longo do tempo;
- Identificar **períodos e regiões críticos**.

### Perguntas orientadoras
1. Quais estados apresentam maior produção agrícola?
2. Quais culturas possuem maior produtividade?
3. Houve crescimento da produção ao longo do tempo?
4. Existe relação entre clima e produtividade?
5. Quais regiões concentram maior área plantada?
6. Quais culturas apresentam maior valor econômico?
7. Existem períodos críticos de produção?

### Tecnologias
Python · Pandas · NumPy · Plotly · SQLite (exportação opcional da base) · Streamlit (dashboard) · GitHub (versionamento)

### Estrutura do notebook
| # | Seção |
|---|-------|
| 1 | Introdução |
| 2 | Contextualização do agronegócio |
| 3 | Explicação da base |
| 4 | Leitura dos dados |
| 5 | Limpeza e preparação |
| 6 | Engenharia de atributos |
| 7 | KPIs |
| 8 | Visualizações |
| 9 | Interpretação |
| 10 | Conclusão |

## 2. Contextualização do agronegócio

O agronegócio é um dos pilares da economia brasileira. Seu desempenho influencia diretamente:

- **Exportações** — soja, café, milho, algodão e açúcar estão entre os principais produtos da pauta exportadora do país;
- **Geração de empregos** — ao longo de toda a cadeia, do campo à indústria e à logística;
- **Abastecimento alimentar** — culturas como arroz e feijão compõem a base da alimentação da população;
- **Crescimento econômico** — participação relevante no PIB e na balança comercial;
- **Desenvolvimento regional** — polos produtores dinamizam economias locais, em especial no interior.

A produção agrícola depende de vários fatores combinados: **clima**, **área plantada**, **produtividade**, **logística** e **demanda de mercado**. Analisar dados agrícolas ajuda a revelar padrões de produção, regiões mais produtivas e tendências do setor.

## 3. Explicação da base

O arquivo `simulacao_producao_agricola_brasil.csv` contém registros mensais de produção agrícola por **estado** e **cultura**, de janeiro/2015 a dezembro/2024. Cada linha é um registro de uma cultura em um estado em um mês.

| Coluna | Descrição | Tipo esperado |
|---|---|---|
| `ano` | Ano da safra | inteiro |
| `mes` | Mês da safra (1–12) | inteiro |
| `data` | Data de referência (primeiro dia do mês) | data |
| `regiao` | Região do Brasil | categórica |
| `uf` | Estado (sigla) | categórica |
| `cultura` | Soja, milho, café etc. | categórica |
| `area_plantada_ha` | Área plantada, em hectares | numérico |
| `producao_toneladas` | Produção total, em toneladas | numérico |
| `produtividade` | Produção por hectare (t/ha) | numérico |
| `chuva_mm` | Volume de chuva, em mm | numérico |
| `temperatura_media` | Temperatura média, em °C | numérico |
| `valor_producao` | Valor econômico da produção, em R$ | numérico |
| `exportacoes` | Volume exportado | numérico |
| `nivel_produtividade` | Classificação: Baixo, Médio, Alto e Muito Alto| categórica ordinal |

## 4. Leitura dos dados

Realizamos o import das bibliotecas e configuração o ambiente. Em seguida localizamos o CSV. O notebook funciona tanto dentro da pasta `notebooks/` do projeto (usando `../dados/`) quanto em outros ambientes.

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pio.templates.default = "plotly_white"
px.defaults.height = 460

In [ ]:
NOME_ARQUIVO = "simulacao_producao_agricola_brasil.csv"

candidatos = [
    Path("..") / "dados" / NOME_ARQUIVO,
    Path("dados") / NOME_ARQUIVO,  
    Path(NOME_ARQUIVO),      
    Path("/mnt/user-data/uploads") / NOME_ARQUIVO,
]
CAMINHO = next((c for c in candidatos if c.exists()), None)
if CAMINHO is None:
    raise FileNotFoundError(
        f"Não encontrei '{NOME_ARQUIVO}'."
    )

RAIZ = CAMINHO.resolve().parent.parent if CAMINHO.parent.name == "dados" else Path.cwd()
PASTA_IMG = RAIZ / "imagens"
PASTA_DB = RAIZ / "database"

df_raw = pd.read_csv(CAMINHO, encoding="utf-8-sig")

In [ ]:
display(df_raw.head(10))
df_raw.info()

In [ ]:
display(df_raw.describe().T)

## 5. Limpeza e preparação

Etapas executadas:
1. Padronização de nomes de colunas e textos;
2. Conversão de tipos (datas e categorias);
3. Verificação de valores nulos, duplicados e valores não positivos;
4. Verificação de **consistência** (data × ano/mês, UF × região, categorias);
5. Detecção de outliers (método IQR) — apenas para **diagnóstico**, sem remoção automática;
6. Checagens de **qualidade analítica** (relações esperadas entre as variáveis).

In [ ]:
df = df_raw.copy()

df.columns = df.columns.str.strip().str.lower()
for col in ["regiao", "uf", "cultura", "nivel_produtividade"]:
    df[col] = df[col].astype(str).str.strip()
df["uf"] = df["uf"].str.upper()

df["data"] = pd.to_datetime(df["data"], errors="coerce")

NUMERICAS = ["area_plantada_ha", "producao_toneladas", "produtividade",
             "chuva_mm", "temperatura_media", "valor_producao", "exportacoes"]

print("Tipos após conversão:")
print(df.dtypes.to_string())

In [ ]:

resumo_qualidade = pd.DataFrame({
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(2),
    "valores únicos": df.nunique(),
})
display(resumo_qualidade)

print(f"Linhas duplicadas: {df.duplicated().sum()}")
print("\nValores <= 0 por coluna numérica:")
print((df[NUMERICAS] <= 0).sum().to_string())

In [ ]:

consistencia = {
    "data coincide com ano/mês": bool(
        ((df["data"].dt.year == df["ano"]) & (df["data"].dt.month == df["mes"])).all()
    ),
    "cada UF pertence a uma única região": bool(df.groupby("uf")["regiao"].nunique().max() == 1),
    "anos de 2015 a 2024": sorted(df["ano"].unique().tolist()) == list(range(2015, 2025)),
    "meses de 1 a 12": sorted(df["mes"].unique().tolist()) == list(range(1, 13)),
}
for k, v in consistencia.items():
    print(("✅" if v else "❌"), k)

print("\nCategorias:")
for col in ["regiao", "cultura", "nivel_produtividade"]:
    print(f"  {col}: {sorted(df[col].unique())}")

print("\nMapa UF → região:")
print(df.groupby("regiao")["uf"].unique().apply(sorted).to_string())

print("\nRegistros por mês de referência (mín / máx):",
      df.groupby("data").size().min(), "/", df.groupby("data").size().max())

In [ ]:
def contar_outliers(serie: pd.Series) -> int:
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)).sum())

outliers = pd.Series({c: contar_outliers(df[c]) for c in NUMERICAS}, name="outliers (IQR)")
display(outliers.to_frame().assign(**{"% do total": (outliers / len(df) * 100).round(2)}))

**Leitura:** a base não possui nulos nem duplicatas, e as colunas são consistentes entre si. Os poucos outliers se concentram em `chuva_mm` e `temperatura_media` (caudas de uma distribuição simulada) e representam valores plausíveis — por isso **não são removidos**.

In [ ]:
prod_calc = df["producao_toneladas"] / df["area_plantada_ha"]

display(df.groupby("nivel_produtividade")["produtividade"]
          .agg(["min", "mean", "max", "count"]).round(2))

### ⚠️ Alertas de qualidade da simulação

As duas checagens acima mostram **limitações estruturais** do dataset simulado, que precisam ser levadas em conta na interpretação:

1. **`produtividade` não é igual a `producao_toneladas ÷ area_plantada_ha`.** A correlação entre as duas é praticamente nula. Ou seja, a coluna `produtividade` foi gerada de forma independente. Neste notebook, **usamos a coluna `produtividade` original** nos KPIs (para manter coerência com o dashboard e com o enunciado) e criamos `produtividade_calc` (produção ÷ área) como atributo adicional para comparação.
2. **`nivel_produtividade` não segue faixas de `produtividade`.** Todos os níveis (Baixo, Médio, Alto, Muito Alto) cobrem praticamente o mesmo intervalo de t/ha. Portanto, o nível funciona aqui como um **rótulo categórico independente**, não como uma classificação derivada da produtividade.

Esses pontos são esperados em dados sintéticos gerados de forma aleatória, e valem como um bom exemplo de por que **validar a consistência interna** dos dados antes de interpretar resultados.

## 6. Engenharia de atributos

Criamos variáveis derivadas para enriquecer as análises:

| Atributo | Definição | Uso |
|---|---|---|
| `nome_mes` | Nome abreviado do mês | rótulos de gráficos |
| `trimestre` | Trimestre do ano (1–4) | sazonalidade |
| `estacao` | Estação do ano (hemisfério sul) | sazonalidade |
| `produtividade_calc` | `producao_toneladas / area_plantada_ha` | verificação da produtividade informada |
| `valor_por_tonelada` | `valor_producao / producao_toneladas` | valor agregado por cultura |
| `valor_por_ha` | `valor_producao / area_plantada_ha` | retorno por área |
| `taxa_exportacao` | `exportacoes / producao_toneladas` | intensidade exportadora |
| `faixa_chuva` / `faixa_temperatura` | Classes de clima | análise clima × produtividade |
| `nivel_produtividade` | Convertida para categoria **ordenada** | ordenação correta em gráficos |

In [ ]:
MESES = {1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr", 5: "Mai", 6: "Jun",
         7: "Jul", 8: "Ago", 9: "Set", 10: "Out", 11: "Nov", 12: "Dez"}
ESTACAO_SUL = {12: "Verão", 1: "Verão", 2: "Verão",
               3: "Outono", 4: "Outono", 5: "Outono",
               6: "Inverno", 7: "Inverno", 8: "Inverno",
               9: "Primavera", 10: "Primavera", 11: "Primavera"}
ORDEM_NIVEL = ["Baixo", "Médio", "Alto", "Muito Alto"]
ORDEM_MESES = list(MESES.values())

df["nome_mes"] = pd.Categorical(df["mes"].map(MESES), categories=ORDEM_MESES, ordered=True)
df["trimestre"] = df["data"].dt.quarter
df["estacao"] = df["mes"].map(ESTACAO_SUL)

df["produtividade_calc"] = df["producao_toneladas"] / df["area_plantada_ha"]
df["valor_por_tonelada"] = df["valor_producao"] / df["producao_toneladas"]
df["valor_por_ha"] = df["valor_producao"] / df["area_plantada_ha"]
df["taxa_exportacao"] = df["exportacoes"] / df["producao_toneladas"]

df["faixa_chuva"] = pd.cut(
    df["chuva_mm"], bins=[0, 50, 100, 150, 200, np.inf],
    labels=["0–50", "50–100", "100–150", "150–200", ">200"], include_lowest=True)
df["faixa_temperatura"] = pd.cut(
    df["temperatura_media"], bins=[0, 20, 24, 28, 32, np.inf],
    labels=["<20 °C", "20–24 °C", "24–28 °C", "28–32 °C", ">32 °C"], include_lowest=True)

df["nivel_produtividade"] = pd.Categorical(df["nivel_produtividade"], categories=ORDEM_NIVEL, ordered=True)

novas = ["nome_mes", "trimestre", "estacao", "produtividade_calc", "valor_por_tonelada",
         "valor_por_ha", "taxa_exportacao", "faixa_chuva", "faixa_temperatura"]
display(df[novas].head())
display(df[["produtividade_calc", "valor_por_tonelada", "valor_por_ha", "taxa_exportacao"]].describe().T)
print(f"Registros com taxa de exportação > 1: {(df['taxa_exportacao'] > 1).mean():.1%} "
      "(possível em dados simulados; a unidade de `exportacoes` não é especificada)")

### Funções auxiliares

Definimos funções reutilizáveis: formatação de números no padrão brasileiro, **filtros** equivalentes aos da barra lateral do dashboard, uma função de **tabela dinâmica** e um utilitário para exibir/salvar gráficos em `imagens/`.

In [ ]:

def br(v, dec=0):
    s = f"{v:,.{dec}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def abrev(v, dec=2):
    for lim, suf in [(1e12, " tri"), (1e9, " bi"), (1e6, " mi"), (1e3, " mil")]:
        if abs(v) >= lim:
            return f"{br(v / lim, dec)}{suf}"
    return br(v, dec)

def filtrar(base, ano=None, mes=None, regiao=None, uf=None, cultura=None, nivel=None):
    """Cada argumento aceita um valor único ou uma lista; None = sem filtro."""
    def como_lista(x):
        return None if x is None else ([x] if isinstance(x, (str, int)) else list(x))

    d = base
    for coluna, valores in [("ano", ano), ("mes", mes), ("regiao", regiao),
                            ("uf", uf), ("cultura", cultura), ("nivel_produtividade", nivel)]:
        valores = como_lista(valores)
        if valores:
            d = d[d[coluna].isin(valores)]
    return d

def tabela_dinamica(base, linhas, colunas=None, valor="producao_toneladas", agg="sum", total=True):
    return base.pivot_table(index=linhas, columns=colunas, values=valor, aggfunc=agg,
                            fill_value=0, margins=total, margins_name="Total", observed=True)

_aviso_img = {"mostrado": False}
def mostrar(fig, nome=None):
    fig.update_layout(margin=dict(t=60, b=40, l=40, r=20))
    fig.show()
    if nome:
        try:
            PASTA_IMG.mkdir(parents=True, exist_ok=True)
            fig.write_image(PASTA_IMG / f"{nome}.png", scale=2)
        except Exception:
            if not _aviso_img["mostrado"]:
                print("ℹ️ Para salvar os gráficos em imagens/, instale: pip install kaleido")
                _aviso_img["mostrado"] = True

## 7. KPIs

Os seis indicadores exigidos no enunciado, mais três complementares (exportações, chuva e temperatura médias). A função `calcular_kpis` aceita qualquer recorte da base — assim os mesmos KPIs podem ser calculados para qualquer combinação de filtros, como no dashboard.

**Definições** (idênticas às do `app.py`):
- **Cultura mais produzida:** maior soma de `producao_toneladas`;
- **Estado mais produtivo:** maior **média** de `produtividade` (t/ha);
- **Média de produtividade:** média simples de `produtividade`.

In [ ]:
def calcular_kpis(d):
    por_cultura = d.groupby("cultura")["producao_toneladas"].sum().sort_values(ascending=False)
    por_uf_prod = d.groupby("uf")["produtividade"].mean().sort_values(ascending=False)
    por_uf_vol = d.groupby("uf")["producao_toneladas"].sum().sort_values(ascending=False)
    return {
        "Produção total (t)": d["producao_toneladas"].sum(),
        "Cultura mais produzida": por_cultura.index[0],
        "Estado mais produtivo (t/ha médio)": por_uf_prod.index[0],
        "Área plantada total (ha)": d["area_plantada_ha"].sum(),
        "Valor econômico total (R$)": d["valor_producao"].sum(),
        "Média de produtividade (t/ha)": d["produtividade"].mean(),
        # complementares
        "Estado com maior produção total": por_uf_vol.index[0],
        "Exportações totais": d["exportacoes"].sum(),
        "Chuva média (mm)": d["chuva_mm"].mean(),
        "Temperatura média (°C)": d["temperatura_media"].mean(),
    }

kpis = calcular_kpis(df)

def formatar_kpi(nome, v):
    if isinstance(v, str):
        return v
    if "R$" in nome or "Produção total" in nome or "Área" in nome or "Exportações" in nome:
        return abrev(v)
    return br(v, 2)

tabela_kpi = pd.DataFrame({
    "KPI": list(kpis.keys()),
    "Valor": [formatar_kpi(k, v) for k, v in kpis.items()],
    "Valor exato": [v if isinstance(v, str) else round(float(v), 2) for v in kpis.values()],
})
display(tabela_kpi)

In [ ]:
destaques = ["Produção total (t)", "Cultura mais produzida", "Estado mais produtivo (t/ha médio)",
             "Área plantada total (ha)", "Valor econômico total (R$)", "Média de produtividade (t/ha)"]
fig = go.Figure()
for i, nome in enumerate(destaques):
    v = kpis[nome]
    fig.add_annotation(
        x=(i % 3) / 2, y=1 - (i // 3) * 0.55, xref="paper", yref="paper",
        text=f"<span style='font-size:13px;color:#555'>{nome}</span><br>"
             f"<b style='font-size:26px;color:#1b5e20'>{formatar_kpi(nome, v)}</b>",
        showarrow=False, align="center")
fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
fig.update_layout(height=230, title="Indicadores-chave de desempenho (2015–2024)",
                  margin=dict(t=60, b=10, l=10, r=10))
mostrar(fig, "00_kpis")

### Filtros interativos (versão notebook)

No dashboard, os filtros ficam na barra lateral. No notebook, a função `filtrar()` reproduz a mesma lógica — todos os critérios obrigatórios (**ano, mês, região, estado, cultura e nível de produtividade**) podem ser combinados. Exemplo: Região Sul, safras 2020–2024, culturas Soja e Milho.

In [ ]:
exemplo = filtrar(df, ano=range(2020, 2025), regiao="Sul", cultura=["Soja", "Milho"])
k = calcular_kpis(exemplo)
display(pd.DataFrame({"KPI": list(k.keys()),
                      "Valor": [formatar_kpi(n, v) for n, v in k.items()]}))

## 8. Visualizações

Esta seção cobre as visualizações obrigatórias do enunciado: **linha temporal**, **barras por cultura**, **barras por estado**, **heatmap sazonal**, **dispersão chuva × produtividade** e **tabela dinâmica** — além de análises complementares (crescimento, regiões, exportações e períodos críticos).

### 8.1 Evolução temporal da produção

In [ ]:
serie_mensal = df.groupby("data")["producao_toneladas"].sum().rename("producao").to_frame()
serie_mensal["media_movel_12m"] = serie_mensal["producao"].rolling(12).mean()

fig = px.line(serie_mensal.reset_index(), x="data", y=["producao", "media_movel_12m"],
              title="Produção mensal total e média móvel de 12 meses",
              labels={"data": "Data", "value": "Produção (t)", "variable": ""})
fig.data[0].name = "Produção mensal"; fig.data[1].name = "Média móvel 12m"
fig.data[1].line.update(width=3, dash="dash")
mostrar(fig, "01_linha_temporal_mensal")

In [ ]:
anual = (df.groupby("ano")
           .agg(producao=("producao_toneladas", "sum"),
                area=("area_plantada_ha", "sum"),
                valor=("valor_producao", "sum"),
                exportacoes=("exportacoes", "sum"),
                produtividade=("produtividade", "mean"))
           .reset_index())

fig = px.line(anual, x="ano", y="producao", markers=True,
              title="Produção total por ano",
              labels={"ano": "Ano", "producao": "Produção (t)"})
fig.update_xaxes(dtick=1)
mostrar(fig, "02_producao_anual")

### 8.2 Crescimento do setor (2015–2024)

Medimos o crescimento com a variação ano a ano, a **taxa composta anual (CAGR)** e a **tendência linear** (inclinação da reta ajustada).

In [ ]:
anual["var_producao_%"] = anual["producao"].pct_change() * 100
anual["var_valor_%"] = anual["valor"].pct_change() * 100
anual["var_exportacoes_%"] = anual["exportacoes"].pct_change() * 100
display(anual.round(2))

def cagr(serie):
    n = len(serie) - 1
    return (serie.iloc[-1] / serie.iloc[0]) ** (1 / n) - 1

def tendencia(x, y):
    """Retorna a inclinação da reta ajustada (por ano) e o R² do ajuste."""
    inc, _ = np.polyfit(x, y, 1)
    r = np.corrcoef(x, y)[0, 1]
    return inc, r ** 2

resumo_crescimento = []
for nome, col in [("Produção (t)", "producao"), ("Valor (R$)", "valor"),
                  ("Exportações", "exportacoes"), ("Área plantada (ha)", "area")]:
    inc, r2 = tendencia(anual["ano"], anual[col])
    resumo_crescimento.append({
        "Indicador": nome,
        "2015": anual[col].iloc[0], "2024": anual[col].iloc[-1],
        "Variação total %": (anual[col].iloc[-1] / anual[col].iloc[0] - 1) * 100,
        "CAGR % a.a.": cagr(anual[col]) * 100,
        "Tendência/ano": inc, "R² da tendência": r2,
    })
resumo_crescimento = pd.DataFrame(resumo_crescimento)
display(resumo_crescimento)

In [ ]:
fig = go.Figure()
fig.add_bar(x=anual["ano"], y=anual["producao"], name="Produção (t)", marker_color="#66bb6a")
inc, _ = np.polyfit(anual["ano"], anual["producao"], 1)
b = anual["producao"].mean() - inc * anual["ano"].mean()
fig.add_scatter(x=anual["ano"], y=inc * anual["ano"] + b, mode="lines",
                name="Tendência linear", line=dict(color="#1b5e20", width=3, dash="dash"))
fig.update_layout(title="Produção anual e tendência linear", xaxis=dict(dtick=1),
                  yaxis_title="Produção (t)")
mostrar(fig, "03_tendencia_producao")

In [ ]:
d = df.groupby(["ano", "cultura"])["producao_toneladas"].sum().reset_index()
fig = px.line(d, x="ano", y="producao_toneladas", color="cultura", markers=True,
              title="Produção anual por cultura",
              labels={"ano": "Ano", "producao_toneladas": "Produção (t)", "cultura": "Cultura"})
fig.update_xaxes(dtick=1)
mostrar(fig, "04_producao_anual_por_cultura")

### 8.3 Análise por cultura

In [ ]:
culturas = (df.groupby("cultura")
              .agg(registros=("cultura", "size"),
                   producao=("producao_toneladas", "sum"),
                   area=("area_plantada_ha", "sum"),
                   produtividade_media=("produtividade", "mean"),
                   valor=("valor_producao", "sum"),
                   exportacoes=("exportacoes", "sum"),
                   valor_por_tonelada=("valor_por_tonelada", "median"))
              .sort_values("producao", ascending=False))
culturas["% da produção"] = culturas["producao"] / culturas["producao"].sum() * 100
culturas["% do valor"] = culturas["valor"] / culturas["valor"].sum() * 100
display(culturas)

In [ ]:
def barras_h(d, x, y, titulo, rotulo_x, nome, cor=None, escala="Greens"):
    d = d.sort_values(x)
    fig = px.bar(d, x=x, y=y, orientation="h", title=titulo, color=cor or x,
                 color_continuous_scale=escala, text_auto=".3s",
                 labels={x: rotulo_x, y: ""})
    fig.update_layout(coloraxis_showscale=False)
    mostrar(fig, nome)

c = culturas.reset_index()
barras_h(c, "producao", "cultura", "Ranking de culturas por produção total", "Produção (t)", "05_cultura_producao")
barras_h(c, "produtividade_media", "cultura", "Produtividade média por cultura", "Produtividade média (t/ha)", "06_cultura_produtividade")
barras_h(c, "valor", "cultura", "Valor econômico por cultura", "Valor (R$)", "07_cultura_valor", escala="YlOrBr")

In [ ]:
part = culturas[["% da produção", "% do valor"]].reset_index().melt(
    id_vars="cultura", var_name="indicador", value_name="percentual")
fig = px.bar(part, x="cultura", y="percentual", color="indicador", barmode="group",
             title="Participação de cada cultura na produção e no valor (%)",
             labels={"percentual": "% do total", "cultura": "Cultura", "indicador": ""})
fig.add_hline(y=100 / len(culturas), line_dash="dot", annotation_text="Participação igual (1/7)")
mostrar(fig, "08_cultura_participacao")

### 8.4 Comparação regional e ranking de estados

In [ ]:
estados = (df.groupby(["uf", "regiao"])
             .agg(registros=("uf", "size"),
                  producao=("producao_toneladas", "sum"),
                  producao_media_registro=("producao_toneladas", "mean"),
                  area=("area_plantada_ha", "sum"),
                  produtividade_media=("produtividade", "mean"),
                  valor=("valor_producao", "sum"),
                  exportacoes=("exportacoes", "sum"))
             .reset_index()
             .sort_values("producao", ascending=False))
estados["ranking_producao"] = range(1, len(estados) + 1)
estados["ranking_produtividade"] = estados["produtividade_media"].rank(ascending=False).astype(int)
display(estados)

In [ ]:
def barras_estado(coluna, titulo, rotulo, nome):
    d = estados.sort_values(coluna)
    fig = px.bar(d, x=coluna, y="uf", color="regiao", orientation="h", title=titulo,
                 labels={coluna: rotulo, "uf": "UF", "regiao": "Região"}, height=620)
    mostrar(fig, nome)

barras_estado("producao", "Produção total por estado (ranking de volume)", "Produção (t)", "09_estado_producao")
barras_estado("produtividade_media", "Produtividade média por estado", "Produtividade média (t/ha)", "10_estado_produtividade")
barras_estado("producao_media_registro", "Produção média por registro (comparação justa entre estados)", "Produção média por registro (t)", "11_estado_producao_media")

In [ ]:
regioes = (df.groupby("regiao")
             .agg(registros=("regiao", "size"),
                  producao=("producao_toneladas", "sum"),
                  area=("area_plantada_ha", "sum"),
                  produtividade_media=("produtividade", "mean"),
                  valor=("valor_producao", "sum"),
                  exportacoes=("exportacoes", "sum"))
             .sort_values("area", ascending=False))
regioes["% da área"] = regioes["area"] / regioes["area"].sum() * 100
regioes["% da produção"] = regioes["producao"] / regioes["producao"].sum() * 100
display(regioes)

r = regioes.reset_index()
c1 = px.pie(r, names="regiao", values="area", hole=0.45, title="Participação das regiões na área plantada")
mostrar(c1, "12_regiao_area")
c2 = px.pie(r, names="regiao", values="producao", hole=0.45, title="Participação das regiões na produção")
mostrar(c2, "13_regiao_producao")

In [ ]:
fig = px.scatter(r, x="area", y="valor", size="producao", color="regiao",
                 title="Área plantada × valor econômico por região (tamanho = produção)",
                 labels={"area": "Área plantada (ha)", "valor": "Valor (R$)", "regiao": "Região"},
                 size_max=60)
mostrar(fig, "14_regiao_area_valor")

d = df.groupby(["ano", "regiao"])["producao_toneladas"].sum().reset_index()
fig = px.line(d, x="ano", y="producao_toneladas", color="regiao", markers=True,
              title="Produção anual por região",
              labels={"ano": "Ano", "producao_toneladas": "Produção (t)", "regiao": "Região"})
fig.update_xaxes(dtick=1)
mostrar(fig, "15_regiao_evolucao")

### 8.5 Heatmap sazonal

In [ ]:
heat = df.pivot_table(index="nome_mes", columns="ano", values="producao_toneladas",
                      aggfunc="sum", fill_value=0, observed=True)
fig = px.imshow(heat, aspect="auto", color_continuous_scale="Greens", text_auto=".2s",
                title="Heatmap sazonal da produção (mês × ano)",
                labels={"x": "Ano", "y": "Mês", "color": "Produção (t)"})
fig.update_xaxes(type="category")
mostrar(fig, "16_heatmap_sazonal")

sazonal = df.groupby("nome_mes", observed=True)["producao_toneladas"].sum().reset_index()
fig = px.bar(sazonal, x="nome_mes", y="producao_toneladas", title="Produção acumulada por mês do ano",
             labels={"nome_mes": "Mês", "producao_toneladas": "Produção (t)"},
             color="producao_toneladas", color_continuous_scale="Greens")
fig.update_layout(coloraxis_showscale=False)
mostrar(fig, "17_sazonalidade_mes")

estacoes = (df.groupby("estacao")["producao_toneladas"].sum()
              .reindex(["Verão", "Outono", "Inverno", "Primavera"]))
display((estacoes / estacoes.sum() * 100).round(2).rename("% da produção").to_frame())

### 8.6 Relação clima × produtividade

In [ ]:
vars_clima = ["chuva_mm", "temperatura_media", "produtividade", "produtividade_calc", "producao_toneladas"]
corr_p = df[vars_clima].corr(method="pearson")
corr_s = df[vars_clima].corr(method="spearman")

display(corr_p.round(3))

fig = px.imshow(corr_p, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Matriz de correlação (Pearson)")
mostrar(fig, "18_matriz_correlacao")

resumo_corr = pd.DataFrame({
    "Pearson": [df["chuva_mm"].corr(df["produtividade"]), df["temperatura_media"].corr(df["produtividade"])],
    "Spearman": [corr_s.loc["chuva_mm", "produtividade"], corr_s.loc["temperatura_media", "produtividade"]],
}, index=["Chuva × produtividade", "Temperatura × produtividade"])
display(resumo_corr.round(4))

In [ ]:
fig = px.scatter(df, x="chuva_mm", y="produtividade", color="cultura", opacity=0.55,
                 hover_data=["ano", "mes", "uf"],
                 title="Dispersão: chuva × produtividade",
                 labels={"chuva_mm": "Chuva (mm)", "produtividade": "Produtividade (t/ha)", "cultura": "Cultura"})
inc, inter = np.polyfit(df["chuva_mm"], df["produtividade"], 1)
xs = np.linspace(df["chuva_mm"].min(), df["chuva_mm"].max(), 50)
fig.add_scatter(x=xs, y=inc * xs + inter, mode="lines", name="Tendência linear",
                line=dict(color="black", width=3, dash="dash"))
mostrar(fig, "19_dispersao_chuva_produtividade")

fig = px.scatter(df, x="temperatura_media", y="produtividade", color="cultura", opacity=0.55,
                 hover_data=["ano", "mes", "uf"],
                 title="Dispersão: temperatura × produtividade",
                 labels={"temperatura_media": "Temperatura média (°C)", "produtividade": "Produtividade (t/ha)", "cultura": "Cultura"})
inc, inter = np.polyfit(df["temperatura_media"], df["produtividade"], 1)
xs = np.linspace(df["temperatura_media"].min(), df["temperatura_media"].max(), 50)
fig.add_scatter(x=xs, y=inc * xs + inter, mode="lines", name="Tendência linear",
                line=dict(color="black", width=3, dash="dash"))
mostrar(fig, "20_dispersao_temperatura_produtividade")

In [ ]:
chuva_faixa = df.groupby("faixa_chuva", observed=True)["produtividade"].agg(["mean", "count"]).reset_index()
temp_faixa = df.groupby("faixa_temperatura", observed=True)["produtividade"].agg(["mean", "count"]).reset_index()

fig = px.bar(chuva_faixa, x="faixa_chuva", y="mean", text="count",
             title="Produtividade média por faixa de chuva (rótulo = nº de registros)",
             labels={"faixa_chuva": "Chuva (mm)", "mean": "Produtividade média (t/ha)"})
mostrar(fig, "21_produtividade_faixa_chuva")

fig = px.bar(temp_faixa, x="faixa_temperatura", y="mean", text="count",
             title="Produtividade média por faixa de temperatura (rótulo = nº de registros)",
             labels={"faixa_temperatura": "Temperatura", "mean": "Produtividade média (t/ha)"})
mostrar(fig, "22_produtividade_faixa_temperatura")

corr_cultura = pd.DataFrame({
    cultura: {"chuva × produtividade": g["chuva_mm"].corr(g["produtividade"]),
              "temperatura × produtividade": g["temperatura_media"].corr(g["produtividade"])}
    for cultura, g in df.groupby("cultura")
}).T
display(corr_cultura.round(3))

### 8.7 Nível de produtividade

In [ ]:
niveis = df["nivel_produtividade"].value_counts().reindex(ORDEM_NIVEL).reset_index()
niveis.columns = ["nivel", "registros"]
fig = px.pie(niveis, names="nivel", values="registros", hole=0.45,
             title="Distribuição dos registros por nível de produtividade",
             category_orders={"nivel": ORDEM_NIVEL})
mostrar(fig, "23_nivel_distribuicao")

nivel_regiao = (pd.crosstab(df["regiao"], df["nivel_produtividade"], normalize="index") * 100).round(2)
display(nivel_regiao)
fig = px.bar(nivel_regiao.reset_index().melt(id_vars="regiao", var_name="nível", value_name="%"),
             x="regiao", y="%", color="nível", barmode="stack",
             category_orders={"nível": ORDEM_NIVEL},
             title="Composição de níveis de produtividade por região (%)",
             labels={"regiao": "Região"})
mostrar(fig, "24_nivel_por_regiao")

### 8.8 Exportações

In [ ]:
fig = px.line(anual, x="ano", y="exportacoes", markers=True, title="Evolução das exportações por ano",
              labels={"ano": "Ano", "exportacoes": "Exportações"})
fig.update_xaxes(dtick=1)
mostrar(fig, "25_exportacoes_anual")

exp_cultura = (df.groupby("cultura")
                 .agg(exportacoes=("exportacoes", "sum"), producao=("producao_toneladas", "sum"))
                 .assign(taxa_exportacao=lambda x: x["exportacoes"] / x["producao"])
                 .sort_values("exportacoes").reset_index())
fig = px.bar(exp_cultura, x="exportacoes", y="cultura", orientation="h", title="Exportações por cultura",
             labels={"exportacoes": "Exportações", "cultura": ""}, text_auto=".3s")
mostrar(fig, "26_exportacoes_cultura")

exp_ano_cultura = df.groupby(["ano", "cultura"])["exportacoes"].sum().reset_index()
fig = px.line(exp_ano_cultura, x="ano", y="exportacoes", color="cultura", markers=True,
              title="Exportações anuais por cultura", labels={"ano": "Ano", "exportacoes": "Exportações", "cultura": "Cultura"})
fig.update_xaxes(dtick=1)
mostrar(fig, "27_exportacoes_ano_cultura")

display(exp_cultura.sort_values("taxa_exportacao", ascending=False).round(3))

### 8.9 Períodos e regiões críticos

In [ ]:
s = serie_mensal["producao"]
z = (s - s.mean()) / s.std()
criticos = s[z <= -1.5]
print(f"Média mensal: {abrev(s.mean())} t | desvio-padrão: {abrev(s.std())} t")
print(f"Meses críticos (z ≤ −1,5): {len(criticos)} de {len(s)}")

fig = px.line(s.reset_index(), x="data", y="producao", title="Meses críticos de produção (z-score ≤ −1,5)",
              labels={"data": "Data", "producao": "Produção (t)"})
fig.add_hline(y=s.mean(), line_dash="dot", annotation_text="Média")
fig.add_hline(y=s.mean() - 1.5 * s.std(), line_dash="dash", line_color="red", annotation_text="Limite crítico")
fig.add_scatter(x=criticos.index, y=criticos.values, mode="markers", name="Mês crítico",
                marker=dict(color="red", size=10))
mostrar(fig, "28_meses_criticos")

piores = (s.nsmallest(10).rename("producao").to_frame()
            .assign(z_score=z.loc[s.nsmallest(10).index]))
piores.index = [f"{MESES[d.month]}/{d.year}" for d in piores.index]
print("\n10 meses de menor produção:")
display(piores)

print("Anos de menor produção:")
display(anual.nsmallest(3, "producao")[["ano", "producao", "var_producao_%"]])

In [ ]:
reg_ano = df.pivot_table(index="regiao", columns="ano", values="produtividade", aggfunc="mean")
fig = px.imshow(reg_ano, aspect="auto", color_continuous_scale="RdYlGn", text_auto=".1f",
                title="Produtividade média (t/ha) por região e ano",
                labels={"x": "Ano", "y": "Região", "color": "t/ha"})
fig.update_xaxes(type="category")
mostrar(fig, "29_produtividade_regiao_ano")

crit = df.groupby("regiao").agg(
    produtividade_media=("produtividade", "mean"),
    pct_nivel_baixo=("nivel_produtividade", lambda x: (x == "Baixo").mean() * 100),
    chuva_media=("chuva_mm", "mean"),
    temperatura_media=("temperatura_media", "mean"),
)
crit["rank_produtividade (1 = pior)"] = crit["produtividade_media"].rank().astype(int)
crit["rank_nivel_baixo (1 = pior)"] = crit["pct_nivel_baixo"].rank(ascending=False).astype(int)
crit["score_criticidade (menor = mais crítica)"] = (
    crit["rank_produtividade (1 = pior)"] + crit["rank_nivel_baixo (1 = pior)"])
display(crit.sort_values("score_criticidade (menor = mais crítica)"))

print("Cinco estados de menor produtividade média:")
display(estados.nsmallest(5, "produtividade_media")[["uf", "regiao", "produtividade_media", "registros"]])

### 8.10 Tabela dinâmica

In [ ]:
pv = tabela_dinamica(df, linhas="uf", colunas="cultura", valor="producao_toneladas", agg="sum")
display(pv.round(0))

fig = px.imshow(tabela_dinamica(df, "uf", "cultura", total=False), aspect="auto",
                color_continuous_scale="Greens", title="Heatmap: produção por estado e cultura",
                labels={"x": "Cultura", "y": "UF", "color": "Produção (t)"})
mostrar(fig, "30_heatmap_estado_cultura")

In [ ]:
display(tabela_dinamica(df, "regiao", "ano", valor="produtividade", agg="mean").round(2))

### 8.11 Respostas às perguntas orientadoras (calculadas)

In [ ]:
top_uf = estados.head(5)
top_prod_uf = estados.nlargest(3, "produtividade_media")
r_cresc = resumo_crescimento.set_index("Indicador").loc["Produção (t)"]
r_chuva = df["chuva_mm"].corr(df["produtividade"])
r_temp = df["temperatura_media"].corr(df["produtividade"])
reg_area = regioes.index[0]

print("1) Estados com maior produção total:")
print("   " + ", ".join(f"{u} ({abrev(p)} t)" for u, p in zip(top_uf["uf"], top_uf["producao"])))
print("   Estados mais produtivos (t/ha médio): "
      + ", ".join(f"{u} ({br(p, 2)})" for u, p in zip(top_prod_uf["uf"], top_prod_uf["produtividade_media"])))

print("\n2) Culturas de maior produtividade média:")
print("   " + ", ".join(f"{c} ({br(p, 2)})" for c, p in culturas["produtividade_media"].sort_values(ascending=False).head(3).items()))
amp = culturas["produtividade_media"].max() - culturas["produtividade_media"].min()
print(f"   Amplitude entre a maior e a menor: {br(amp, 2)} t/ha")

print(f"\n3) Crescimento da produção 2015→2024: {br(r_cresc['Variação total %'], 2)}% no total "
      f"({br(r_cresc['CAGR % a.a.'], 2)}% a.a.); R² da tendência linear = {br(r_cresc['R² da tendência'], 2)}")

print(f"\n4) Clima × produtividade: correlação chuva = {r_chuva:.3f}; temperatura = {r_temp:.3f}")

print(f"\n5) Região com maior área plantada: {reg_area} "
      f"({br(regioes.loc[reg_area, '% da área'], 1)}% da área total)")

print("\n6) Culturas de maior valor econômico:")
print("   " + ", ".join(f"{c} (R$ {abrev(v)})" for c, v in culturas['valor'].sort_values(ascending=False).head(3).items()))

print(f"\n7) Períodos críticos: {len(criticos)} mês(es) com z ≤ −1,5; pior mês: "
      f"{s.idxmin():%m/%Y} ({abrev(s.min())} t); pior ano: {int(anual.loc[anual['producao'].idxmin(), 'ano'])}")

### 8.12 Exportação da base tratada para SQLite (opcional)

In [ ]:
try:
    PASTA_DB.mkdir(parents=True, exist_ok=True)
    caminho_db = PASTA_DB / "prod_agricula.db"
    base_sql = df_raw.copy()  # base original, idêntica à usada pelo dashboard
    with sqlite3.connect(caminho_db) as conn:
        base_sql.to_sql("prod_agricola", conn, if_exists="replace", index=False)
        consulta = pd.read_sql(
            """
            SELECT cultura,
                   ROUND(SUM(producao_toneladas), 0) AS producao_total,
                   ROUND(AVG(produtividade), 2)      AS produtividade_media
            FROM prod_agricola
            GROUP BY cultura
            ORDER BY producao_total DESC
            """, conn)
    display(consulta)
except Exception as e:
    print("Não foi possível gravar o banco SQLite neste ambiente:", e)

## 9. Interpretação

### 9.1 Panorama geral
O conjunto soma **≈ 1,12 bilhão de toneladas** produzidas em **≈ 178,2 milhões de hectares**, com valor econômico de **≈ R$ 1,10 trilhão**, produtividade média de **40,44 t/ha** e **≈ 554,6 milhões** em exportações.

### 9.2 Culturas
- **Mais produzida:** **Feijão** (≈ 167,4 mi t), seguido de perto por **Soja** (≈ 167,1 mi t) e **Arroz** (≈ 165,6 mi t). A menos produzida, Cana-de-açúcar, tem 13,5% do total contra 15,0% do Feijão — a **diferença é pequena** (cerca de 1,5 ponto percentual). A liderança de Feijão sobre Soja é praticamente um empate técnico.
- **Maior valor econômico:** **Soja** (≈ R$ 166,2 bi), à frente de Feijão (R$ 162,7 bi) e Algodão (R$ 158,4 bi).
- **Maior produtividade média:** **Algodão** (40,86 t/ha) e **Café** (40,85 t/ha); a menor é Cana-de-açúcar (39,92 t/ha). A amplitude entre extremos é de apenas **0,94 t/ha (≈ 2%)**, ou seja, as culturas são praticamente equivalentes em produtividade nesta base.

### 9.3 Estados e regiões
- **Maior volume:** RJ (≈ 124,3 mi t), SP, ES e MG — todos do **Sudeste**. Porém, **RJ possui 480 registros** (o dobro de estados como PA ou BA, e o quádruplo de MS ou MT). Como a produção total é uma soma, o ranking de volume reflete em boa parte a **quantidade de registros** de cada estado. Na produção **média por registro**, o líder é **MS** (≈ 277,5 mil t), e RJ fica em torno de 258,9 mil t.
- **Mais produtivos (t/ha médio):** **MS (43,08)**, **AM (43,07)** e **TO (42,98)**. Os menos produtivos são **BA (37,48)**, SP (38,74) e GO (38,81).
- **Área plantada:** o **Sudeste concentra 35,8%** da área (e 35,2% da produção), seguido do Nordeste (20,6%) e do Sul (16,7%). Centro-Oeste (13,8%) e Norte (13,2%) têm menor área.
- **Produtividade regional:** **Norte** (41,75 t/ha) e **Sul** (41,12) lideram; o **Sudeste** tem a menor (39,72). Ou seja, **a região com mais área não é a mais produtiva**.

### 9.4 Evolução temporal
- A produção passou de **110,3 mi t (2015)** para **116,6 mi t (2024)**: **+5,8%** no período, ou **≈ 0,6% ao ano** (CAGR). O pico foi em **2023 (117,5 mi t)**.
- A tendência linear é positiva, mas **fraca/moderada** (R² ≈ 0,31) e a série oscila de ano para ano (de −3,0% a +6,1%). Há crescimento, porém **modesto e irregular**.
- O **valor econômico** variou apenas **+1,2%** entre as pontas e as **exportações** **−1,5%**, sem tendência clara (R² baixíssimo). A taxa exportação/produção é semelhante entre culturas (≈ 0,46 a 0,53).
- **Sazonalidade:** novembro é o mês de maior produção acumulada (≈ 98,3 mi t) e maio o de menor (≈ 88,5 mi t). A primavera concentra 26,0% da produção, contra 24,4–24,8% nas demais estações — um padrão **sutil**, sem a forte sazonalidade típica de safras reais.

### 9.5 Clima × produtividade
As correlações de Pearson são **praticamente nulas**: chuva × produtividade ≈ **0,007** e temperatura × produtividade ≈ **−0,002** (Spearman confirma). Dentro de cada cultura, nenhuma passa de |0,06|. Os gráficos de dispersão formam uma nuvem sem direção e a produtividade média é parecida em todas as faixas de chuva e temperatura. **Não há evidência de relação clima–produtividade nesta base** — resultado coerente com dados gerados aleatoriamente, mas que *não* deve ser lido como ausência de relação no mundo real, onde clima é determinante.

### 9.6 Períodos e regiões críticos
- **Meses críticos** (produção ≥ 1,5 desvio-padrão abaixo da média): **9 de 120**. Os piores foram **jun/2020** (7,36 mi t; z = −2,32), **mai/2020** e **mai/2021** (z ≈ −2,10).
- **Pior ano:** **2021** (107,6 mi t; −3,0% sobre 2020).
- **Regiões críticas:** o **Sudeste** tem a menor produtividade média e a 2ª maior proporção de registros "Baixo"; o **Centro-Oeste** tem a maior proporção de nível "Baixo" (30,8%) e a série mais volátil (de 33,5 t/ha em 2018 a 45,4 t/ha em 2020). Os demais ficam em torno de 25–26% de nível "Baixo".

### 9.7 Limitações
1. Dados **simulados**: padrões reais (safra, clima, preços) não estão refletidos.
2. `produtividade` **não é** `produção ÷ área` (correlação ≈ 0,01) e `nivel_produtividade` **não** acompanha a produtividade — ver seção 5.
3. Número de registros **desigual entre estados** (120 a 480), o que distorce rankings baseados em soma.
4. `exportacoes` não tem unidade definida e em ≈ 25% dos registros excede a produção.
5. Diferenças entre culturas e entre regiões são **pequenas**; sem testes estatísticos, não é seguro afirmar que sejam significativas.

## 10. Conclusão

### Conclusão executiva

| Pergunta | Resposta (base simulada 2015–2024) |
|---|---|
| Estados com maior produção | **RJ, SP, ES e MG** em volume (efeito parcial do maior nº de registros); **MS** é o mais produtivo (43,08 t/ha) |
| Culturas mais produtivas | **Algodão e Café** (≈ 40,9 t/ha), mas com diferença de ≈ 2% para a última |
| Crescimento ao longo do tempo | **Sim, modesto:** +5,8% em 9 anos (≈ 0,6% a.a.), com oscilações |
| Relação clima × produtividade | **Não detectada** (r ≈ 0,00 para chuva e temperatura) |
| Região com maior área plantada | **Sudeste** (35,8% do total) |
| Cultura de maior valor econômico | **Soja** (≈ R$ 166,2 bi) |
| Períodos críticos | **2021** (pior ano) e **mai–jun/2020**, **mai/2021** (piores meses) |